# Training methodology — Sinhala Document OCR

This notebook is the **methodology** companion to the project: how the recogniser
was trained. It walks the pipeline in depth — page deskew and projection
detection, line-crop preprocessing (`prepare_line_*`), the CRNN in
`src/recognition/model.py`, and the CTC training loop in
`src/recognition/train.py` — then dataset counts, logs and held-out metrics.
It is written for a review panel / dissertation chapter, not as the short
inference demo.

The delivered demo notebooks (`local_pipeline.ipynb`, `colab_pipeline.ipynb`)
**only load** `models/crnn_best.pth` and run inference. They do not retrain the
model. Full prose lives in [`docs/Methodology.md`](../docs/Methodology.md);
headline numbers in [`RESULTS.md`](../RESULTS.md).

**Tone of this work.** We train a single general CRNN+CTC model for *printed*
Sinhala pages. Tamil and handwriting are out of scope. Poem line crops appear
below only as a **training-mix** source — this notebook is not a live OCR demo
of that page.



## 0. Environment (Colab Drive mount or local)

On Google Colab, set `DRIVE_PROJECT_DIR` to the Drive folder that contains this
repository (same pattern as `colab_pipeline.ipynb`). Locally, `IN_COLAB` is
false and the notebook uses the repo root. A missing gitignored checkpoint is
**not** fatal here: counts, configs and curves still run. The optional train
cell at the end is off by default (`RUN_TRAIN = False`).



In [ ]:
# Folder on Google Drive that contains the project (edit if needed).
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/sinhala-document-ocr"
DRIVE_ZIP_PATH = ""  # optional: unzip once into DRIVE_PROJECT_DIR

import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path(DRIVE_PROJECT_DIR)
    if DRIVE_ZIP_PATH and Path(DRIVE_ZIP_PATH).is_file() and not (root / "src").is_dir():
        root.mkdir(parents=True, exist_ok=True)
        !unzip -o -q "{DRIVE_ZIP_PATH}" -d "{root}"
        kids = [p for p in root.iterdir() if p.is_dir() and (p / "src").exists()]
        if kids and not (root / "src").exists():
            root = kids[0]
    REPO_ROOT = root.resolve()
else:
    here = Path.cwd().resolve()
    if (here / "training_methodology.ipynb").exists():
        REPO_ROOT = here.parent
    elif (here / "notebooks" / "training_methodology.ipynb").exists():
        REPO_ROOT = here
    else:
        # Walk up looking for configs/local.yaml
        REPO_ROOT = here
        for p in [here, *here.parents]:
            if (p / "configs" / "local.yaml").is_file():
                REPO_ROOT = p
                break

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("IN_COLAB:", IN_COLAB)
print("REPO_ROOT:", REPO_ROOT)
ckpt = REPO_ROOT / "models" / "crnn_best.pth"
print("checkpoint present:", ckpt.is_file(), "|", ckpt)


## Setup

Install project requirements when needed (Colab). Locally this is a no-op if
the environment already has PyTorch and the rest of `requirements.txt`.



In [ ]:
%pip install -q -r requirements.txt
if IN_COLAB:
    !apt-get -qq update && apt-get -qq install -y fonts-noto-core fonts-noto-ui-core >/dev/null

import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from matplotlib import font_manager as fm

from src.utils.display import configure_display_utf8, setup_matplotlib_sinhala

configure_display_utf8()
setup_matplotlib_sinhala()
try:
    from src.utils.display import ensure_latin_plot_fonts
    ensure_latin_plot_fonts()
except Exception:
    _dejavu = fm.findfont(fm.FontProperties(family="DejaVu Sans"))
    plt.rcParams["font.family"] = "DejaVu Sans"
    plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
    try:
        fm.fontManager.addfont(_dejavu)
    except Exception:
        pass

print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 1. Introduction and objectives

Optical Character Recognition for printed Sinhala is harder than for Latin
scripts because of a large grapheme inventory, pre-base vowel signs (kombuva)
that sit visually *between* consonants, Zero-Width Joiner conjuncts, and a
shortage of large publicly labelled page datasets. Phone photographs add skew,
JPEG compression and lines that may be only 12–26 px tall after cropping.

This work implements a full **page → text** pipeline and trains **one general**
CRNN with CTC on a consumer GPU (developed on an RTX 4060 laptop). The
objectives are:

1. Detect text lines on photographed or scanned pages (deskew + projection profile).
2. Recognise each line with a CRNN trained on synthetic data *and* a modest real mix.
3. Decode with CTC beam search plus a character language model, then apply only
   those orthographic post-corrections that do not regress held-out sets.
4. Report **end-to-end** CER/WER with a hard split between **held-out** evidence
   and **in-training** reference scores.

The methodological claim we do *not* make: synthetic validation CER, or near-zero
CER on poem / `user_batch1` crops that the trainer saw, is **not**
generalisation.



## 2. System architecture (detect → recognise)

```
page image → deskew / enhance → projection line boxes → line crops
         → CRNN (CNN + BiLSTM) → CTC decode (greedy | beam | beam+LM)
         → sinhala_fix post-correct → page transcription
```

| Stage | Module | Design choice |
|---|---|---|
| Deskew | `src/detection/text_detection.py` | Projection-profile sharpness over ±5°. A small skew merges adjacent lines on a photographed column. |
| Detection | same; `detection.method: projection` | Contrast-binarised ink mask, border/watermark suppression, tall-band re-split at the internal valley. Exact on the eval suite (76/76, 9/9, 23/23). |
| Recognition | `src/recognition/model.py` | CRNN, height 48 px, CTC over 224 characters + blank. |
| Decode | `src/recognition/decode.py` | Default `beam_lm`, `lm_weight=0.2`. |
| Post-correct | `src/postprocess/sinhala_fix.py` | Measured rules only; no retrain for the final accuracy push. |
| Eval path | `src/evaluation/pipeline_eval.py` | Shared by CLI, notebooks and `run_eval_suite.py`. |

**Why not a learned detector (DBNet/CRAFT) as the delivered path?** A classical
projection detector is inspectable, trains no extra weights, and was exact on
every page we score. A neural detector remains an optional future path, not
the examined delivery.

**Why CRNN+CTC rather than TrOCR?** CTC removes character boxes; the stack fits
one GPU and the available labelled budget. Transformer OCR is noted in the
repository as an alternative, not as the trained artefact.



## 3. Datasets

We combine three kinds of data:

1. **Synthetic line crops** — the only way to cover 224 classes from scratch.
2. **Detector-in-the-loop page crops** — so training sees the same imperfect
   boxes inference produces.
3. **A small real mix** (user pages, exam-cover lines, CC-BY-4.0 Acts, and ten
   literary poem lines used *only* as extra-labels).

The cell below counts label files on disk. Gitignored trees
(`data/synthetic*/`) may be absent on a fresh clone; in that case we print the
documented count from `RESULTS.md` / `docs/Methodology.md` instead of crashing.



In [ ]:
from typing import Optional

DOCUMENTED = {
    # Documented fallbacks when gitignored label files are missing.
    "data/synthetic/train_labels.txt": 21000,
    "data/synthetic/val_labels.txt": 4500,
    "data/synthetic/test_labels.txt": 4500,
    "data/synthetic_pages/train_labels.txt": 21713,
    "data/synthetic_pages/val_labels.txt": 3832,
    "data/synthetic_hard/train_labels.txt": 12000,
    "data/synthetic_small/train_labels.txt": 10000,
    "data/synthetic_small2/train_labels.txt": 10000,
    "data/real/labels/poem_kanyawee.txt": 10,
    "data/real/labels/poem_kanyawee_aug.txt": 810,
    "data/real/labels/user_batch1.txt": 91,
    "data/real/labels/user_batch1_aug.txt": 3731,
    "data/real/labels/user_batch1_holdout.txt": 41,
    "data/real/labels/web_batch1.txt": 6,
    "data/real/labels/web_batch1_aug.txt": 486,
    "data/real/labels/web_batch1_holdout.txt": 14,
    "data/real/labels/web_batch1_acts.txt": 2275,
    "data/real/labels/web_batch1_acts_aug.txt": 9100,
    "data/real/labels/web_batch1_acts_extra.txt": 1895,
}


def count_label_rows(path: Path) -> Optional[int]:
    if not path.is_file():
        return None
    n = 0
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        s = line.strip()
        if s and not s.startswith("#"):
            n += 1
    return n


def report_label(rel: str, role: str, status: str) -> dict:
    path = REPO_ROOT / rel
    n = count_label_rows(path)
    documented = DOCUMENTED.get(rel)
    if n is None:
        note = f"not present; documented {documented} lines in RESULTS.md / Methodology.md"
        shown = documented
        source = "documented"
    else:
        note = "on disk"
        shown = n
        source = "disk"
        if documented is not None and n != documented:
            note = f"on disk ({n}); documented figure was {documented}"
    return {"file": rel, "n": shown, "source": source, "role": role, "status": status, "note": note}


rows = [
    report_label("data/synthetic/train_labels.txt", "primary synthetic train (70%)", "train (generator)"),
    report_label("data/synthetic/val_labels.txt", "synthetic val (15%) — trainer metric", "trainer split"),
    report_label("data/synthetic/test_labels.txt", "synthetic test (15%)", "generator test"),
    report_label("data/synthetic_pages/train_labels.txt", "detector-in-the-loop extra-labels", "in mix"),
    report_label("data/synthetic_pages/val_labels.txt", "page-synth val split", "not the real holdout"),
    report_label("data/synthetic_hard/train_labels.txt", "hard styles (pill/dark/serif/tiny)", "in mix"),
    report_label("data/synthetic_small/train_labels.txt", "all-tiny 11–26 px (Jul-28)", "in mix"),
    report_label("data/synthetic_small2/train_labels.txt", "second tiny set (Jul-29)", "NOT in delivered model"),
    report_label("data/real/labels/poem_kanyawee.txt", "literary poem lines (training mix only)", "IN TRAINING"),
    report_label("data/real/labels/poem_kanyawee_aug.txt", "poem ×80 geometric/photometric aug", "IN TRAINING"),
    report_label("data/real/labels/user_batch1.txt", "real user page line crops", "IN TRAINING"),
    report_label("data/real/labels/user_batch1_aug.txt", "user_batch1 augmented", "IN TRAINING"),
    report_label("data/real/labels/user_batch1_holdout.txt", "historical name; later folded into mix", "IN TRAINING (not a holdout)"),
    report_label("data/real/labels/web_batch1.txt", "exam-cover / web crops", "IN TRAINING"),
    report_label("data/real/labels/web_batch1_aug.txt", "web_batch1 augmented", "IN TRAINING"),
    report_label("data/real/labels/web_batch1_holdout.txt", "14 hard-style crops", "PARTIAL LEAK (6/14)"),
    report_label("data/real/labels/web_batch1_acts.txt", "HF Acts train pages (CC-BY-4.0)", "IN TRAINING"),
    report_label("data/real/labels/web_batch1_acts_aug.txt", "Acts light aug", "IN TRAINING"),
    report_label("data/real/labels/web_batch1_acts_extra.txt", "HF Acts test split extras", "reserved; not in Jul-28 train"),
]

print(f"{'n':>7}  {'src':<11}  {'status':<32}  file")
print("-" * 110)
for r in rows:
    print(f"{r['n'] or 0:7d}  {r['source']:<11}  {r['status']:<32}  {r['file']}")
    if r["source"] != "disk":
        print(f"         {r['note']}")

# Jul-28 merged volume (delivered continue-train extras + synthetic cap)
mix = {
    "synthetic cap (mix_jul28.synthetic_train_max)": 10000,
    "page-synth train": rows[3]["n"],
    "hard": rows[5]["n"],
    "tiny": rows[6]["n"],
    "poem aug": rows[9]["n"],
    "user aug": rows[11]["n"],
    "web aug": rows[14]["n"],
    "acts aug": rows[17]["n"],
    "acts raw": rows[16]["n"],
}
print("\nJul-28 merged rows (documented mix; extras that exist on disk):")
total = 0
for k, v in mix.items():
    v = int(v or 0)
    total += v
    print(f"  {v:7d}  {k}")
print(f"  {total:7d}  TOTAL  (RESULTS.md states ~70k)")


### Why each source exists

**Synthetic lines.** There are not enough labelled real Sinhala lines to train
a 225-class CTC model from scratch. `scripts/generate_data.py` renders lines
from `src/data/corpus_sinhala.txt` (~3,269 sentences; `corpus_ratio: 0.65`)
plus word recombinations, numbers and mixed Sinhala–English, using every
available Sinhala font face and SynthTIGER-style degradations (rotation, blur,
noise, JPEG, shadow, defocus, paper texture, rare moiré, edge artefacts).
Config split: **70 / 15 / 15** → 21,000 / 4,500 / 4,500 of 30,000.

**Detector-in-the-loop pages.** A tightly cropped rendered line is *not* what
inference sees. The projection detector returns padded, occasionally clipped
or slightly merged bands. `scripts/generate_pages.py` renders a full page
(paragraph / bordered card / poem / mixed / letterhead), runs the **same**
detector, and keeps crops only when the detected line count matches ground
truth. Mismatched pages are discarded, not mislabelled. On this machine:
25,545 images = 21,713 train + 3,832 val.

**Hard and tiny lines.** Real exam covers and lyric cards failed in ways
generic blur does not explain: dark/pill titles, serif book print, and 11–26 px
crops. `scripts/generate_hard_lines.py` targets those styles. The Jul-28
`synthetic_small` set crushes *every* line to 11–26 px, half left small so the
dataset performs the same upscale that inference does (`pad_to_height: false`).

**Real mix.** `user_batch1` (91 labelled lines), six exam-cover lines, CC-BY-4.0
Sri Lankan Acts pages from Hugging Face (2,275 detector-matched lines), and
ten literary poem lines. The poem set is a **training mix** only — we do not
treat its 0.00 CER as a demo of general OCR. All 41 transcripts in
`user_batch1_holdout.txt` later entered training; the filename is historical.

**Held-out eval (next cell).** The only fully clean real photographs are
`data/eval_real/print_photos/` (2 pages, 32 lines).



In [ ]:
def count_gt_lines(folder: Path):
    # Return (n_images, n_gt_lines) for a page folder with sidecar .gt.txt files.
    if not folder.is_dir():
        return 0, 0
    imgs = list(folder.glob("*.png")) + list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg"))
    n_lines = 0
    for gt in folder.glob("*.gt.txt"):
        n_lines += sum(1 for ln in gt.read_text(encoding="utf-8").splitlines() if ln.strip())
    return len(imgs), n_lines


eval_sets = [
    ("data/eval_real/print_photos", "real photographed pages", "HELD OUT (headline)", 2, 32),
    ("data/eval_pages", "synthetic pages, 5 layouts, separate generation run", "HELD OUT", 10, 76),
    ("data/eval_real/adversarial", "hand-built stress pages (card / article / photo)", "HELD OUT (synthetic)", 3, 26),
]

print(f"{'pages':>6} {'lines':>6}  {'src':<8}  status")
print("-" * 88)
for rel, why, status, doc_p, doc_l in eval_sets:
    folder = REPO_ROOT / rel
    n_p, n_l = count_gt_lines(folder)
    if n_p == 0 and n_l == 0:
        print(f"{doc_p:6d} {doc_l:6d}  documented  {status}  {rel}")
        print(f"         not present on disk — {why}")
    else:
        print(f"{n_p:6d} {n_l:6d}  disk      {status}  {rel}")
        print(f"         {why}")

corpus = REPO_ROOT / "src" / "data" / "corpus_sinhala.txt"
if corpus.is_file():
    n_corpus = sum(1 for ln in corpus.read_text(encoding="utf-8").splitlines() if ln.strip())
    print(f"\nCorpus sentences (src/data/corpus_sinhala.txt): {n_corpus}")


## 4. Page preprocessing and line detection

The delivered path is **not** a learned detector. A photographed page is
deskewed, converted to a contrast-binarised ink mask, then split into line
boxes by a horizontal projection profile. The same functions are used at
training time (detector-in-the-loop page crops) and at evaluation
(`src/evaluation/pipeline_eval.py`).

**Deskew** (`src/detection/text_detection.py`). Let \(M_\theta\) be the ink
mask rotated by \(\theta\) degrees. The sharpness of the horizontal profile is

\[
S(\theta) = \sum_y \left(\sum_x M_\theta(x,y)\right)^2
\]

`estimate_page_skew` searches \(\theta \in [-5^\circ, +5^\circ]\) (coarse
step \(0.5^\circ\), then fine \(0.1^\circ\)). The page is rotated only when
\(S(\theta^\star) \ge 1.02\,S(0)\) and \(|\theta^\star| \ge 0.3^\circ\)
(`detection.deskew_min_angle`). A few degrees of uncorrected skew merge
adjacent projection bands on a photographed column.

**Binarise for detection.** `binarize_for_detection` estimates a local
background with a large median blur, takes darkness relative to that
background, thresholds with Otsu, and floors the threshold at
`min_contrast: 25` so faint watermarks stay out of the ink mask. Decorative
frames are then erased by `suppress_border_structures`.

**ProjectionLineDetector.** Horizontal ink profile \(\to\) text bands
(rows with ink fraction \(\ge\) `min_ink_frac`) \(\to\) merge only
diacritic fragments \(\to\) re-split tall bands at an internal valley
\(\to\) per-band ink-extent box. Config: `detection.method: projection`.
The cell below runs this on the held-out lyric photograph when present,
otherwise on a synthetic bar page.



In [ ]:
import cv2
import numpy as np

from src.detection.text_detection import (
    ProjectionLineDetector,
    binarize_for_detection,
    build_detector,
    crop_lines,
    estimate_page_skew,
    rotate_page,
)
from src.utils.common import load_config

cfg_local = load_config(str(REPO_ROOT / "configs" / "local.yaml"))
det_cfg = cfg_local.get("detection", {})
img_cfg = cfg_local.get("image", {})
inf_cfg = cfg_local.get("inference", {})

print("detection config:")
for k in ("method", "deskew", "deskew_max_angle", "deskew_min_angle",
          "min_contrast", "suppress_borders", "min_ink_frac",
          "min_ink_density", "min_height_frac", "crop_padding_x",
          "crop_padding_y", "min_crop_height"):
    print(f"  {k}: {det_cfg.get(k)}")

page_path = REPO_ROOT / "data" / "eval_real" / "print_photos" / "page_song_lyrics.jpg"
demo_source = str(page_path)
if page_path.is_file():
    bgr = cv2.imread(str(page_path), cv2.IMREAD_COLOR)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
else:
    demo_source = "synthetic fallback (page_song_lyrics.jpg not on disk)"
    gray = np.full((400, 640), 240, dtype=np.uint8)
    for i, y in enumerate((40, 90, 140, 200, 260, 320)):
        gray[y:y + 18, 40:600 - i * 8] = 30
    bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

print("page source:", demo_source)
print("gray shape (H, W):", gray.shape, "| dtype:", gray.dtype, "| mean:", f"{float(gray.mean()):.1f}")

detector = build_detector(det_cfg)
print("detector class:", type(detector).__name__)
assert isinstance(detector, ProjectionLineDetector) or det_cfg.get("method") != "projection"

mask_before = detector.ink_mask(gray) if hasattr(detector, "ink_mask") else binarize_for_detection(gray)
angle = estimate_page_skew(mask_before, max_angle=float(det_cfg.get("deskew_max_angle", 5.0)))
min_ang = float(det_cfg.get("deskew_min_angle", 0.3))
applied = abs(angle) >= min_ang
gray_upright = rotate_page(gray, angle) if applied else gray
print(f"estimated skew: {angle:.2f} deg | applied: {applied} (min {min_ang} deg)")

boxes = detector.detect(gray_upright)
crops = crop_lines(gray_upright, boxes, padding=det_cfg)
print(f"detected line boxes: {len(boxes)}")
for i, (x, y, w, h) in enumerate(boxes[:8], start=1):
    print(f"  line {i:02d}: x={x:4d} y={y:4d} w={w:4d} h={h:3d}")
if len(boxes) > 8:
    print(f"  ... {len(boxes) - 8} more")

overlay = cv2.cvtColor(gray_upright, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in boxes:
    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 255), 2)
mask_after = detector.ink_mask(gray_upright) if hasattr(detector, "ink_mask") else binarize_for_detection(gray_upright)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("1. Input (grayscale)")
axes[1].imshow(mask_after, cmap="gray")
axes[1].set_title("2. Ink mask (binarise + border suppress)")
axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"3. Projection boxes (n={len(boxes)})")
for ax in axes:
    ax.axis("off")
fig.suptitle("Page preprocessing and ProjectionLineDetector", fontsize=12)
plt.tight_layout()
plt.show()

DEMO_GRAY = gray_upright
DEMO_CROPS = crops
DEMO_BOXES = boxes
print("stored DEMO_CROPS:", len(DEMO_CROPS))


## 5. Line-crop recognition preprocessing

Each detector crop is mapped to the CRNN input by
`prepare_line_for_recognition` / `prepare_line_tensor`
(`src/recognition/inference.py`). Training (`OCRLineDataset`) uses the same
geometry: grayscale, height **48**, max width **512**, pixels scaled to
roughly \([-1, 1]\).

**Steps (inference).**

1. Convert to uint8 grayscale (`_to_gray_uint8`).
2. Optional median denoise (off in the delivered config).
3. Polarity: invert if `mode=="invert"`, or if `mode=="auto"` and
   `should_invert_polarity` (mean intensity \(< 128\)) — light-on-dark titles
   become dark-on-light.
4. Height policy:
   * `pad_to_height: true` — white-pad a short crop to 48 px (glyphs stay small).
   * `pad_to_height: false` (**delivered**) — skip padding; `resize_keep_height`
     *upscales* the crop so glyphs fill 48 px. This matches training.
5. `resize_keep_height`: \(w' = \min(W_{\max},\, \mathrm{round}(w \cdot 48 / h))\).
6. Tensor: \(x \leftarrow x/255\), then \(x \leftarrow (x - 0.5)/0.5\).
   White (\(255\)) becomes \(+1\); that is also the collate pad value.

The train/inference mismatch that used to leave an ~18 px lyric line sitting
inside a 48 px canvas is documented in `RESULTS.md` §3a.



In [ ]:
import inspect

from src.data.dataset import resize_keep_height
from src.recognition.inference import (
    line_image_to_tensor,
    prepare_line_for_recognition,
    prepare_line_tensor,
    prepared_line_for_display,
    should_invert_polarity,
)

print("=== prepare_line_for_recognition (source) ===")
print(inspect.getsource(prepare_line_for_recognition))

height = int(inf_cfg.get("image_height", img_cfg.get("height", 48)))
max_width = int(img_cfg.get("max_width", 512))
pad_to_height = bool(inf_cfg.get("pad_to_height", True))
auto_invert = bool(inf_cfg.get("auto_invert", True))
print("inference.image_height:", height, "| max_width:", max_width)
print("inference.pad_to_height:", pad_to_height, "(False = upscale short crops)")
print("inference.auto_invert:", auto_invert, "| denoise:", inf_cfg.get("denoise"))

if DEMO_CROPS:
    # Prefer a short crop if one exists (lyric-card style), else the middle line.
    heights = [c.shape[0] for c in DEMO_CROPS]
    demo_i = int(np.argmin(heights))
    demo_crop = DEMO_CROPS[demo_i]
else:
    demo_i = -1
    demo_crop = np.full((18, 220), 245, dtype=np.uint8)
    demo_crop[4:14, 10:210] = 25

print(f"\ndemo crop index: {demo_i} | raw shape (H, W): {demo_crop.shape[:2]}")
print("should_invert_polarity:", should_invert_polarity(demo_crop),
      "| mean:", f"{float(np.mean(demo_crop)):.1f}")

prepared_upscale = prepare_line_for_recognition(
    demo_crop, height=height, max_width=max_width,
    mode="auto", auto_invert=auto_invert, pad_to_height=False,
)
prepared_padded = prepare_line_for_recognition(
    demo_crop, height=height, max_width=max_width,
    mode="auto", auto_invert=auto_invert, pad_to_height=True,
)
tensor = prepare_line_tensor(
    demo_crop, height=height, max_width=max_width, channels=1,
    mode="auto", auto_invert=auto_invert, pad_to_height=pad_to_height,
)

print("prepared (upscale, pad_to_height=False) PIL size (W, H):", prepared_upscale.size)
print("prepared (pad_to_height=True) PIL size (W, H):         ", prepared_padded.size)
print("tensor shape (B, C, H, W):", tuple(tensor.shape),
      "| min/mean/max:",
      f"{float(tensor.min()):.3f} / {float(tensor.mean()):.3f} / {float(tensor.max()):.3f}")
print("normalization: x/255 then (x-0.5)/0.5  ->  white pad in collate is +1.0")

fig, axes = plt.subplots(3, 1, figsize=(12, 5))
axes[0].imshow(demo_crop, cmap="gray")
axes[0].set_title(f"Detector crop (raw)  H={demo_crop.shape[0]}  W={demo_crop.shape[1]}")
axes[1].imshow(np.asarray(prepared_padded), cmap="gray")
axes[1].set_title(f"pad_to_height=True (old path)  size={prepared_padded.size}  — glyphs stay short")
axes[2].imshow(np.asarray(prepared_upscale), cmap="gray")
axes[2].set_title(f"pad_to_height=False (delivered)  size={prepared_upscale.size}  — upscale to {height} px")
for ax in axes:
    ax.set_yticks([])
    ax.set_xticks([])
fig.suptitle("Line-crop recognition preprocessing", fontsize=12)
plt.tight_layout()
plt.show()


## 6. CRNN architecture

The recogniser is a Convolutional Recurrent Neural Network (Shi, Bai & Yao,
*IEEE TPAMI* 2017), implemented in `src/recognition/model.py`. Nothing is
invented beyond that file: a CNN backbone, an AdaptiveAvgPool to height 1,
a map-to-sequence permute, two `BidirectionalLSTM` blocks, and a linear CTC
head. `forward` returns **log-probabilities** of shape `(T, B, num_classes)`
for `nn.CTCLoss`.

**CNN** (`CRNN.cnn`), input `(B, 1, 48, W)`:

| Block | Layers | Spatial note |
|---|---|---|
| 1 | Conv 3×3, 1→64, ReLU; MaxPool 2×2 | H/2, W/2 |
| 2 | Conv 3×3, 64→128, ReLU; MaxPool 2×2 | H/4, W/4 |
| 3 | Conv 3×3, 128→256, ReLU; Conv 3×3, 256→256, ReLU; MaxPool (2,2) stride (2,1) pad (0,1) | H/8, W/4 |
| 4 | Conv 3×3, 256→512, **BN**, ReLU; Conv 3×3, 512→512, **BN**, ReLU; MaxPool (2,2) stride (2,1) pad (0,1) | H/16, W/4 |
| 5 | Conv 2×2, 512→512, **BN**, ReLU (stride 1, pad 0) | height collapses further |

`conv_bn(..., bn=False)` is Conv+ReLU only. BatchNorm appears only on the
512-channel blocks and the final 2×2 convolution — matching `model.py`.

**Map-to-sequence.** If feature height is not 1, `AdaptiveAvgPool2d((1, None))`
collapses it. Then squeeze height and `permute(2, 0, 1)`: `(B, C', W') → (T, B, C')`
with \(T = W'\).

**Recurrent head.** Two stacked `BidirectionalLSTM` modules:

1. LSTM(`cnn_out_channels=512` → hidden 256, **2 layers**, bidirectional, dropout 0.1) + Linear(512 → 256)
2. LSTM(256 → hidden 256, **1 layer**, bidirectional) + Linear(512 → `num_classes`)

`rnn_layers: 2` in the config is the layer count of the *first* LSTM block,
not “two extra stacks”. The second block always uses the class default
`num_layers=1`.

**Charset.** 224 printable characters in `models/charset.json` plus CTC blank
at index 0 → **225** output classes. Sinhala block U+0D80–U+0DFF, ZWJ/ZWNJ,
ASCII, digits, form punctuation (`src/charset.py`).



In [ ]:
import inspect

from src.charset import Charset
from src.recognition.model import CRNN, build_crnn

cs_path = REPO_ROOT / cfg_local["paths"]["charset_path"]
if cs_path.is_file():
    charset = Charset.load(str(cs_path))
else:
    charset = Charset.build_default()
    print("charset.json missing — using build_default() for the count only.")

print("charset path:", cs_path, "| exists:", cs_path.is_file())
print("printable characters:", len(charset.chars))
print("classes including CTC blank:", charset.num_classes)
print("image.height:", cfg_local["image"]["height"],
      "| max_width:", cfg_local["image"]["max_width"],
      "| channels:", cfg_local["image"]["channels"])
print("model config:", json.dumps(cfg_local.get("model", {}), indent=2))
print("CTC blank index:", Charset.BLANK_INDEX)
sample = [c for c in charset.chars if c.isascii()][:20]
print("first ASCII members:", "".join(sample))
print("ZWJ in charset:", "\u200d" in charset.chars, "| ZWNJ:", "\u200c" in charset.chars)

print("\n=== CRNN source (model.py) ===")
print(inspect.getsource(CRNN))


In [ ]:
model = build_crnn(
    charset.num_classes,
    cfg_local.get("model"),
    in_channels=int(cfg_local["image"]["channels"]),
)
print(model)
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nparameters: {n_params:,}  (trainable {n_trainable:,})")

# Dummy forward: input (B, C, H, W) -> log-probs (T, B, num_classes)
H = int(cfg_local["image"]["height"])
W = 256
B = 2
dummy = torch.zeros(B, int(cfg_local["image"]["channels"]), H, W)
model.eval()
with torch.no_grad():
    log_probs = model(dummy)
print(f"input  (B, C, H, W) = {tuple(dummy.shape)}")
print(f"output (T, B, C)    = {tuple(log_probs.shape)}  "
      f"(T is the CNN width after pooling; C = num_classes = {charset.num_classes})")
print("log_softmax along class dim: row sums of exp ≈ 1:",
      float(log_probs.exp().sum(-1)[0, 0]))

# Show how W maps to T for a few widths (including a prepared demo crop).
print("\nwidth → T (sequence length):")
for w in (64, 128, 256, 512, tensor.shape[-1]):
    x = torch.zeros(1, 1, H, int(w))
    with torch.no_grad():
        y = model(x)
    print(f"  W={int(w):4d}  ->  T={y.shape[0]:3d}  shape {tuple(y.shape)}")


## 7. Training loop (CTC, collate, extra-labels, checkpointing)

Training lives in `src/recognition/train.py`. One epoch is
`train_one_epoch`: for each collated batch, run the CRNN, compute CTC loss,
clip gradients, step Adam. After each epoch the synthetic *val* loader is
decoded greedily and scored with CER; `crnn_best.pth` is overwritten only
when that val CER improves. `crnn_last.pth` is written every epoch.

**CTC.** `nn.CTCLoss(blank=0, zero_infinity=True)` (Graves et al., ICML 2006).
The network emits a length-\(T\) distribution over 225 symbols; the loss
marginalises over all alignments that collapse (drop blanks, merge repeats)
to the Unicode transcript. No character boxes are required.

**Collate** (`src/data/dataset.py` `_ctc_collate_batch`). Images in a batch
have a shared height 48 but different widths. They are right-padded to the
batch max width with pad value **1.0** (normalised white). Targets are
concatenated; `target_lengths` go to CTC; `input_lengths` are all \(T\).

**Extra-labels mix.** `build_dataloader(..., extra_label_paths=...,
max_primary_samples=synthetic_train_max)` concatenates extra label files
after a *capped* synthetic train split. Repeat count is
`train.extra_label_repeat` (1 for Jul-28). That is how page-synth, hard,
tiny, poem-aug, user-aug, web-aug and Acts rows enter the same epoch.

**What “best” means.** The trainer’s bar is **synthetic val CER**. A
domain-mix run can worsen that metric while helping real pages, so the
previous checkpoint is copied first (`crnn_best_pre_jul28.pth`) and the
keep/reject decision is made with `scripts/run_eval_suite.py` on held-out
pages. Resume also seeds `best_cer` from the checkpoint so a worse first
epoch cannot clobber a stronger prior.

`RUN_TRAIN` stays **False** at the end of this notebook. The cell below
prints the loop, the delivered CLI, and the Jul-28 epoch table from the log.



In [ ]:
import inspect

from src.data.dataset import _ctc_collate_batch, _merge_label_records
from src.recognition import train as train_mod

print("=== train_one_epoch ===")
print(inspect.getsource(train_mod.train_one_epoch))

print("=== CTC criterion (from train.main) ===")
print("  criterion = nn.CTCLoss(blank=Charset.BLANK_INDEX, zero_infinity=True)")
print("  Charset.BLANK_INDEX =", Charset.BLANK_INDEX)

print("\n=== collate (_ctc_collate_batch) ===")
print(inspect.getsource(_ctc_collate_batch))

print("=== extra-label merge (_merge_label_records signature) ===")
print(inspect.getsource(_merge_label_records).split("rows: List")[0])


In [ ]:
# Delivered continue-train CLI (configs/mix_jul28.yaml header + Methodology §9).
jul28_cli = """python -m src.recognition.train --config configs/mix_jul28.yaml \\
    --extra-labels data/synthetic_pages/train_labels.txt \\
    --extra-labels data/synthetic_hard/train_labels.txt \\
    --extra-labels data/synthetic_small/train_labels.txt \\
    --extra-labels data/real/labels/poem_kanyawee_aug.txt \\
    --extra-labels data/real/labels/user_batch1_aug.txt \\
    --extra-labels data/real/labels/web_batch1_aug.txt \\
    --extra-labels data/real/labels/web_batch1_acts_aug.txt \\
    --extra-labels data/real/labels/web_batch1_acts.txt \\
    --resume models/crnn_best.pth"""
print("=== CLI that produced models/crnn_best.pth (Jul-28 continue-train) ===")
print(jul28_cli)

# Checkpoint selection as written in train.py (val CER, not held-out pages).
print("""
=== how crnn_best.pth is selected inside train.py ===
  best_cer = resumed_cer or +inf
  each epoch:
    train_one_epoch  ->  train_loss
    evaluate_model(val_loader)  ->  val_cer, val_wer
    save crnn_last.pth
    if val_cer < best_cer:
        best_cer = val_cer
        save crnn_best.pth
    else:
        epochs_without_improvement += 1
        early stop if patience exceeded
    ReduceLROnPlateau.step(val_cer)
Promotion to the *delivered* artefact also required the held-out suite
(scripts/run_eval_suite.py). Jul-29 had better synth val CER and was rejected.
""")

from src.evaluation.train_curves import parse_train_log, summary_table_rows

log_path = REPO_ROOT / "models" / "train_jul28.log"
hist_json = REPO_ROOT / "data" / "metrics" / "train_history_jul28.json"
print("=== Jul-28 epoch table ===")
hist_src = None
if log_path.is_file() and log_path.stat().st_size > 0:
    hist28 = parse_train_log(log_path)
    hist_src = log_path
elif hist_json.is_file():
    from src.evaluation.train_curves import load_train_history
    hist28 = load_train_history(hist_json)
    hist_src = hist_json
else:
    hist28 = None

if hist28 is None:
    print("neither train_jul28.log nor train_history_jul28.json is present.")
else:
    print("source:", hist_src)
    print(f"epochs logged: {len(hist28)} | best synthetic val CER: {hist28.best_val_cer}")
    cols = ["epoch", "train_loss", "val_cer", "val_wer", "lr"]
    rows_h = summary_table_rows(hist28)
    widths = {c: max(len(c), max((len(f"{r.get(c)}") for r in rows_h), default=0)) for c in cols}
    print("  ".join(c.ljust(widths[c]) for c in cols))
    print("  ".join("-" * widths[c] for c in cols))
    for r in rows_h:
        print("  ".join(f"{r.get(c)}".ljust(widths[c]) for c in cols))


## 8. Training chronology and hyperparameters

The delivered file `models/crnn_best.pth` is **not** a from-scratch 12-epoch
run. It is the last *promoted* continue-train (`configs/mix_jul28.yaml`) on a
general CRNN that had already seen synthetic lines, page-synth crops, hard
styles and real mixes.

**Promotion rule.** `src/recognition/train.py` overwrites `crnn_best.pth` when
*synthetic* val CER improves. Domain-mix runs can worsen that metric while
helping real pages, so we copy the previous checkpoint and decide with
`scripts/run_eval_suite.py`. A later Jul-29 run reached **better** synthetic
val CER (0.0343 vs 0.0348) and **worse** every held-out set; it was not
promoted (`RESULTS.md` §3d). Epoch counts below come from parsed logs, not
from config upper bounds.



In [ ]:
from src.evaluation.train_curves import parse_train_log

# Configs actually used. mix_jul28 inherits mix_web → local.
config_files = [
    "configs/local.yaml",
    "configs/mix_real.yaml",
    "configs/mix_web.yaml",
    "configs/mix_jul28.yaml",
    "configs/mix_jul29.yaml",
]

print("=== train.* hyperparameters ===")
for rel in config_files:
    path = REPO_ROOT / rel
    if not path.is_file():
        print(rel, "missing")
        continue
    cfg = load_config(str(path))
    t = cfg.get("train", {})
    keys = ["epochs", "lr", "batch_size", "optimizer", "lr_scheduler",
            "lr_factor", "lr_patience", "early_stopping_patience",
            "synthetic_train_max", "extra_label_repeat", "grad_clip"]
    bits = ", ".join(f"{k}={t.get(k)}" for k in keys if k in t or t.get(k) is not None)
    print(f"\n{rel}")
    print(" ", bits)

print("\n=== epochs actually logged (do not invent these) ===")
log_notes = {
    "train_v2.log": "v2 diverse synthetic (config 40; early stop)",
    "train_jul27.log": "Jul-27 book-serif + tiny continue-train",
    "train_jul28.log": "DELIVERED continue-train (mix_jul28, ~70k rows)",
    "train_jul29.log": "REJECTED (better synth val, worse holdouts)",
    "train_web_batch.log": "web/hard mix (config 25)",
    "train_user_batch1.log": "user_batch1 mix (mix_real, config 15)",
    "train_poem_mix.log": "poem extra-labels mix",
}
for name, note in log_notes.items():
    p = REPO_ROOT / "models" / name
    if not p.is_file() or p.stat().st_size == 0:
        extra = ""
        if name == "train_v2.log":
            extra = " | README v3 from-scratch/continue: 40 epochs, best 0.0311 at epoch 37 (train_v3.log empty)"
        print(f"{name}: not present or empty. {note}{extra}")
        continue
    hist = parse_train_log(p)
    print(
        f"{name}: {len(hist)} epoch(s) logged, "
        f"best synthetic val CER={hist.best_val_cer}, "
        f"last train_loss={hist.train_loss[-1] if hist.train_loss else None}  — {note}"
    )

v3 = REPO_ROOT / "models" / "train_v3.log"
print(
    "\ntrain_v3.log size:",
    v3.stat().st_size if v3.is_file() else "missing",
    "| documented in README: 40 epochs, best val CER 0.0311 at epoch 37, ~4h11m RTX 4060.",
)


### Delivered round (`mix_jul28.yaml`)

| Item | Value |
|---|---|
| Extra-labels | page-synth train + hard + tiny + poem aug + user aug + web aug + acts aug + acts raw |
| Synthetic cap | 10,000 (so extras stay influential) |
| Epochs | **12** (all completed; see log) |
| LR | 8×10⁻⁵, plateau ×0.5 patience 2 → 4×10⁻⁵ from epoch 6 |
| Batch | 32 |
| Best synthetic val CER | **0.0348** at epoch 12 |
| Hardware | RTX 4060 laptop, ~3.5 h |
| Log / history | `models/train_jul28.log`, `data/metrics/train_history_jul28.json` |

CLI (from `docs/Methodology.md` §9): `python -m src.recognition.train --config configs/mix_jul28.yaml` with the `--extra-labels` list and `--resume models/crnn_best.pth`.



## 9. Training curves (saved history)

We plot the **delivered** continue-train. Preference order matches
`src/evaluation/train_curves.py`: `models/train_jul28.log` if it parses, else
`models/train_history.json`, else the bundled
`data/metrics/train_history_jul28.json` (so a clone without logs still shows
the 12-epoch curve).



In [ ]:
from src.evaluation.train_curves import (
    format_eval_summary_table,
    load_eval_summary,
    plot_eval_cer_bars,
    plot_train_curves,
    resolve_train_history,
    summary_table_rows,
)

hist = resolve_train_history(REPO_ROOT)
print("history source:", hist.source)
print(f"epochs: {len(hist)} | best synthetic val CER: {hist.best_val_cer}")
cols = ["epoch", "train_loss", "val_cer", "val_wer", "lr"]
rows_h = summary_table_rows(hist)
widths = {c: max(len(c), max((len(f"{r.get(c)}") for r in rows_h), default=0)) for c in cols}
print("  ".join(c.ljust(widths[c]) for c in cols))
print("  ".join("-" * widths[c] for c in cols))
for r in rows_h:
    print("  ".join(f"{r.get(c)}".ljust(widths[c]) for c in cols))

fig = plot_train_curves(
    hist,
    title="Delivered continue-train (Jul-28, mix_jul28.yaml)",
    show=False,
)
plt.show()


## 10. Decoding and post-processing

**CTC decode.** Default `inference.decode: beam_lm`. A 6-gram character LM
(Witten–Bell) is built at load time from *training-side* Sinhala text only
(`src/postprocess/char_lm.py`). Shallow fusion:

```
score = log P_ctc(y|x) + lm_weight · log P_lm(y) + insertion_bonus · |y|
```

Swept on the delivered checkpoint: `lm_weight=0.2` improved or tied every
held-out set (real photos 0.0990 → 0.0877). Weights ≳ 0.5 overfit the LM
corpus. Greedy decode is the fast path. Test-time augmentation was measured
and left **off** (a trade, not a strict win).

**Why beam+LM?** Residual errors on tiny lyric crops are orthographic
(`ේ`/`ී`, `්`/`ි`), not generic blur. A character LM trained on the project
corpus prefers legal Sinhala sequences without seeing held-out transcripts.

**Post-correct (`inference.post_correct: true`).** `fix_sinhala_ocr` repairs
mis-attached kombuva, a few lyric refrain artefacts, and a gated literary
long-uu — only rules that leave `eval_pages` / `adversarial` unchanged.
Network weights did not change for §3e–3g in `RESULTS.md`. Blind `ස්සු` and
bare `ණි`→`ණේ` were rejected because they regress synthetic holdouts.



In [ ]:
from src.recognition.decode import decode_log_probs
from src.postprocess.sinhala_fix import fix_sinhala_ocr

inf = cfg_local.get("inference", {})
print("inference.decode:", inf.get("decode"))
print("lm_weight:", inf.get("lm_weight"), "| insertion_bonus:", inf.get("insertion_bonus"))
print("beam_width:", inf.get("beam_width"), "| beam_top_k:", inf.get("beam_top_k"),
      "| lm_order:", inf.get("lm_order"))
print("pad_to_height:", inf.get("pad_to_height"),
      "(False = upscale short crops — matches training resize_keep_height)")
print("post_correct:", inf.get("post_correct"), "| tta:", inf.get("tta"))
det = cfg_local.get("detection", {})
print("detection.method:", det.get("method"), "| deskew:", det.get("deskew"),
      "| deskew_max_angle:", det.get("deskew_max_angle"))

print("\n=== decode_log_probs (greedy vs beam vs beam_lm) ===")
print(inspect.getsource(decode_log_probs))
print("=== charset.ctc_greedy_decode ===")
print(inspect.getsource(charset.ctc_greedy_decode))
print("=== fix_sinhala_ocr (first ~40 lines) ===")
src_fix = inspect.getsource(fix_sinhala_ocr)
print("\n".join(src_fix.splitlines()[:40]))
print("  ...")


## 11. Evaluation protocol and final metrics

**Protocol.** Every number that matters is **end-to-end**: deskew → detect →
recognise. Detection mistakes count. Lines are aligned in order against
`.gt.txt`, so a missed line inflates page CER.

**Definitions** (`src/evaluation/metrics.py`):

* CER = Levenshtein on characters / |reference|
* WER = Levenshtein on whitespace tokens / word count(reference)
* Character Accuracy % = (1 − CER) × 100
* Word Accuracy % = (1 − WER) × 100

Corpus metrics sum edits over the whole set (not the mean of per-line rates).

**Leakage.** `scripts/check_holdout_leakage.py` classifies each eval set.
Green bars below are held out; grey bars are in-training reference only.



In [ ]:
summary = load_eval_summary(REPO_ROOT)
print("eval summary source note:")
print(" ", summary.get("source"))
print(" ", summary.get("note"))
print()
print(format_eval_summary_table(summary))

fig2 = plot_eval_cer_bars(
    summary,
    title="Held-out vs in-train CER / Character Accuracy %",
    show=False,
)
plt.show()

print()
print("Headline (held-out real photos): CER 0.0325 → Char Acc 96.75%; "
      "WER 0.1491 → Word Acc 85.09%.")
print("Do not quote poem / user_batch1 ~0% CER as generalisation — those sets are in training.")


### Limitations (honest)

1. Fully held-out **real** data is 2 pages / 32 lines.
2. Synthetic val CER has saturated as a model-selection signal (Jul-29).
3. `ේ`/`ී` on ~16 px crops is reduced, not solved; `ු`/`ූ` on serif book faces
   remains partly wrong.
4. Logos / decorative display fonts are not suppressed.
5. Handwriting is out of scope.

Reproduce the suite:

```powershell
python scripts/check_holdout_leakage.py
python scripts/run_eval_suite.py --checkpoint models/crnn_best.pth
```



## 12. Optional short retrain (off by default)

This cell demonstrates the **train loop** (loss, val CER, checkpoint save). It
is **not** the 12-epoch Jul-28 schedule. `RUN_TRAIN = False` so Restart & Run
All stays a methodology walkthrough.

By default weights are written to `models/crnn_method_demo.pth`, **not**
`models/crnn_best.pth`. Set `REPLACE_DELIVERED_CHECKPOINT = True` only if you
intentionally want the trainer’s usual `crnn_best.pth` path (resume still
refuses to overwrite unless synthetic val CER beats the resumed bar).



In [ ]:
RUN_TRAIN = False
REPLACE_DELIVERED_CHECKPOINT = False  # keep the delivered model unless you opt in
TRAIN_MODE = "short"  # "short" | "full"
TRAIN_CONFIG = "configs/local.yaml"
RESUME_CKPT = "models/crnn_best.pth"
SHORT_EPOCHS = 1
SHORT_SYN_MAX = 256
SHORT_BATCH = 16

if not RUN_TRAIN:
    print("RUN_TRAIN=False — training skipped. This notebook is a methodology walkthrough.")
    print("Set RUN_TRAIN=True for a 1-epoch local/Colab demo (writes crnn_method_demo.pth by default).")
else:
    import subprocess
    import sys

    syn_train = REPO_ROOT / "data" / "synthetic" / "train_labels.txt"
    syn_val = REPO_ROOT / "data" / "synthetic" / "val_labels.txt"
    if not syn_train.is_file() or not syn_val.is_file():
        raise FileNotFoundError(
            "Need data/synthetic/{train,val}_labels.txt. "
            "Generate with: python scripts/generate_data.py --config configs/local.yaml --large"
        )

    resume = REPO_ROOT / RESUME_CKPT
    cmd = [sys.executable, "-m", "src.recognition.train", "--config", TRAIN_CONFIG]
    if resume.is_file():
        cmd += ["--resume", str(resume)]

    if not REPLACE_DELIVERED_CHECKPOINT:
        cmd += [
            "paths.finetune_best=models/crnn_method_demo.pth",
            "paths.finetune_last=models/crnn_method_demo_last.pth",
        ]

    if TRAIN_MODE == "short":
        cmd += [
            f"train.epochs={SHORT_EPOCHS}",
            f"train.synthetic_train_max={SHORT_SYN_MAX}",
            f"train.batch_size={SHORT_BATCH}",
            "train.val_every=1",
            "train.early_stopping_patience=0",
            "train.num_workers=0",
        ]
        print(
            f"Short demo: {SHORT_EPOCHS} epoch(s), synthetic_train_max={SHORT_SYN_MAX}. "
            "Not the delivered Jul-28 schedule."
        )
    else:
        print(f"Full schedule via {TRAIN_CONFIG} (can take hours).")

    if REPLACE_DELIVERED_CHECKPOINT:
        print("WARNING: REPLACE_DELIVERED_CHECKPOINT=True — trainer may update crnn_best.pth.")
    else:
        print("Writing demo weights to models/crnn_method_demo.pth (delivered checkpoint untouched).")

    print("Running:", " ".join(cmd))
    rc = subprocess.call(cmd, cwd=str(REPO_ROOT))
    if rc != 0:
        raise RuntimeError(f"Training failed with exit code {rc}")

    from src.evaluation.train_curves import load_train_history, plot_train_curves, summary_table_rows

    hist_path = REPO_ROOT / "models" / "train_history.json"
    demo = REPO_ROOT / "models" / (
        "crnn_best.pth" if REPLACE_DELIVERED_CHECKPOINT else "crnn_method_demo.pth"
    )
    print("Wrote:", demo, "| exists:", demo.is_file())
    if hist_path.is_file():
        _hist = load_train_history(hist_path)
        print(f"Epochs: {len(_hist)} | best val CER: {_hist.best_val_cer}")
        for r in summary_table_rows(_hist):
            print(r)
        _fig = plot_train_curves(_hist, show=False)
        plt.show()


## Further reading

* [`docs/Methodology.md`](../docs/Methodology.md) — matching prose: preprocessing steps, CRNN diagram, training algorithm, reproduce CLI.
* [`RESULTS.md`](../RESULTS.md) — held-out tables, ablations, rejected levers.
* [`docs/Project_Report.md`](../docs/Project_Report.md) — dissertation-style write-up.
* Demo (inference only): `notebooks/local_pipeline.ipynb` / `notebooks/colab_pipeline.ipynb`.

